# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/clever-dhruv/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [32]:
import os

print(os.listdir("/content"))

['.config', 'flyrank-ml-internship', 'sample_data']


In [33]:
!git clone https://github.com/clever-dhruv/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [34]:
import os

print(os.listdir("/content/flyrank-ml-internship"))

['work', 'outputs', 'README.md', '.github', '.gitignore', 'docs', 'data', 'SETUP.md', 'scripts', 'LICENSE', 'submission', 'DATA_USE.md', 'GUIDE.md', 'skills', 'requirements.txt', 'AGENTS.md', '.git', 'notebooks', 'CLAUDE.md']


In [35]:
print(os.listdir("/content/flyrank-ml-internship/data/raw"))

['content_refresh_anonymized.csv']


In [36]:
import pandas as pd

DATA_PATH = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print(df.head())

Rows: 30000
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     commercial         NaN   
4               LOW  0.00  keyword article  informational      2803.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0  ...     15000-

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## 1. Method choice

"""I will use Logistic Regression as the first ML model for the Content Refresh
Prediction lane.

The task is to identify pages that are showing signs of declining performance
and prioritize them for content-refresh review. Logistic Regression is a good
first model because it is simple, fast, interpretable, and provides a clear
baseline for understanding how the available signals relate to the target.

It is preferable to starting with a more complex model because the goal is
not to maximize complexity. The model should provide a fair comparison against
the Week-4 hand-written baseline and its feature effects should remain
interpretable.

I will use the model to estimate the probability that a page belongs to the
declining-performance class, then rank pages by that probability."""

'I will use Logistic Regression as the first ML model for the Content Refresh\nPrediction lane.\n\nThe task is to identify pages that are showing signs of declining performance\nand prioritize them for content-refresh review. Logistic Regression is a good\nfirst model because it is simple, fast, interpretable, and provides a clear\nbaseline for understanding how the available signals relate to the target.\n\nIt is preferable to starting with a more complex model because the goal is\nnot to maximize complexity. The model should provide a fair comparison against\nthe Week-4 hand-written baseline and its feature effects should remain\ninterpretable.\n\nI will use the model to estimate the probability that a page belongs to the\ndeclining-performance class, then rank pages by that probability.'

In [38]:
MODEL_NAME = "Logistic Regression"

print("Method:", MODEL_NAME)
print("Lane: Content Refresh Prediction")
print("Goal: rank pages for refresh review")

Method: Logistic Regression
Lane: Content Refresh Prediction
Goal: rank pages for refresh review


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## 2. Split design

"""I will use an 80/20 stratified train-test split.

The training set will contain 80% of the pages and the held-out test set will
contain the remaining 20%. Stratification will preserve the proportion of
declining and non-declining pages in both sets.

The test set will not be used during model fitting. It will be used only for
the final comparison between the Logistic Regression model and the Week-4
baseline.

I will use a fixed random seed (`42`) so that the split and results are
reproducible."""

'I will use an 80/20 stratified train-test split.\n\nThe training set will contain 80% of the pages and the held-out test set will\ncontain the remaining 20%. Stratification will preserve the proportion of\ndeclining and non-declining pages in both sets.\n\nThe test set will not be used during model fitting. It will be used only for\nthe final comparison between the Logistic Regression model and the Week-4\nbaseline.\n\nI will use a fixed random seed (`42`) so that the split and results are\nreproducible.'

In [40]:
FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "days_since_last_update",
    "impressions_90d"
]
for col in FEATURES:
    print(col, "->", col in df.columns)

search_volume -> True
competition -> True
cpc -> True
word_count -> True
char_count -> True
ctr -> True
avg_position -> True
engagement_rate -> True
scroll_rate -> True
ai_traffic_pct -> True
days_since_last_update -> True
impressions_90d -> True


In [41]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [42]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print(df["is_declining_label"].value_counts())
print("Declining rate:", round(df["is_declining_label"].mean(), 4))

is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Declining rate: 0.5421


In [43]:
from sklearn.model_selection import train_test_split

TARGET = "is_declining_label"

FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "days_since_last_update",
    "impressions_90d"
]

X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining target rate:", round(y_train.mean(), 4))
print("Test target rate:", round(y_test.mean(), 4))

Training rows: 24000
Test rows: 6000

Training target rate: 0.5421
Test target rate: 0.542


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [44]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## 3. Train + compare against my baseline

"""I will train Logistic Regression using the training split only.

The preprocessing pipeline uses median imputation for missing numerical
values and standardization before Logistic Regression. Keeping preprocessing
inside the pipeline prevents information from the test set from influencing
the training process.

The model will output a probability of declining performance. Pages will be
ranked by this probability.

I will compare the model against the Week-4 rule on the same held-out test
set using ROC-AUC, average precision, precision, recall, and F1. The comparison
will focus on whether the model provides a meaningful improvement rather than
whether it is more complex."""

'I will train Logistic Regression using the training split only.\n\nThe preprocessing pipeline uses median imputation for missing numerical\nvalues and standardization before Logistic Regression. Keeping preprocessing\ninside the pipeline prevents information from the test set from influencing\nthe training process.\n\nThe model will output a probability of declining performance. Pages will be\nranked by this probability.\n\nI will compare the model against the Week-4 rule on the same held-out test\nset using ROC-AUC, average precision, precision, recall, and F1. The comparison\nwill focus on whether the model provides a meaningful improvement rather than\nwhether it is more complex.'

In [45]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [46]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

test_prob = model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

print("ROC-AUC:", round(roc_auc_score(y_test, test_prob), 4))
print("Average Precision:", round(average_precision_score(y_test, test_prob), 4))
print("Precision:", round(precision_score(y_test, test_pred), 4))
print("Recall:", round(recall_score(y_test, test_pred), 4))
print("F1:", round(f1_score(y_test, test_pred), 4))

ROC-AUC: 0.5719
Average Precision: 0.59
Precision: 0.5742
Recall: 0.8324
F1: 0.6796


In [47]:
# Recreate the Week-4 baseline on the exact same test rows

baseline_test = df.loc[X_test.index].copy()

baseline_pred = (
    baseline_test["days_since_last_update"].between(91, 365)
    &
    baseline_test["impressions_90d"].between(300, 29999)
).astype(int)

print("Baseline positive predictions:", baseline_pred.sum())
print("Baseline positive rate:", round(baseline_pred.mean(), 4))

Baseline positive predictions: 1347
Baseline positive rate: 0.2245


In [48]:
baseline_precision = precision_score(y_test, baseline_pred, zero_division=0)
baseline_recall = recall_score(y_test, baseline_pred, zero_division=0)
baseline_f1 = f1_score(y_test, baseline_pred, zero_division=0)

print("Baseline Precision:", round(baseline_precision, 4))
print("Baseline Recall:", round(baseline_recall, 4))
print("Baseline F1:", round(baseline_f1, 4))

Baseline Precision: 0.6214
Baseline Recall: 0.2574
Baseline F1: 0.364


In [49]:
baseline_score = np.where(
    baseline_test["days_since_last_update"].between(91, 365)
    & baseline_test["impressions_90d"].between(300, 29999),
    baseline_test["impressions_90d"],
    0
)

baseline_auc = roc_auc_score(y_test, baseline_score)
baseline_ap = average_precision_score(y_test, baseline_score)

print("Baseline ROC-AUC:", round(baseline_auc, 4))
print("Baseline Average Precision:", round(baseline_ap, 4))

Baseline ROC-AUC: 0.5347
Baseline Average Precision: 0.5573


In [50]:
### Model vs baseline

"""| Metric | Week-4 Baseline | Logistic Regression |
|---|---:|---:|
| ROC-AUC | 0.5347 | 0.5719 |
| Average Precision | 0.5573 | 0.5900 |
| Precision | 0.6214 | 0.5742 |
| Recall | 0.2574 | 0.8324 |
| F1 | 0.3640 | 0.6796 |

Logistic Regression performs better than the Week-4 baseline on ROC-AUC,
Average Precision, recall, and F1 on the same held-out test set. The baseline
has higher precision, meaning that its positive recommendations are more
selective.

The model therefore provides a stronger overall ranking and identifies many
more declining pages, but it also produces more false positives. This is a
trade-off rather than an across-the-board improvement."""

'| Metric | Week-4 Baseline | Logistic Regression |\n|---|---:|---:|\n| ROC-AUC | 0.5347 | 0.5719 |\n| Average Precision | 0.5573 | 0.5900 |\n| Precision | 0.6214 | 0.5742 |\n| Recall | 0.2574 | 0.8324 |\n| F1 | 0.3640 | 0.6796 |\n\nLogistic Regression performs better than the Week-4 baseline on ROC-AUC,\nAverage Precision, recall, and F1 on the same held-out test set. The baseline\nhas higher precision, meaning that its positive recommendations are more\nselective.\n\nThe model therefore provides a stronger overall ranking and identifies many\nmore declining pages, but it also produces more false positives. This is a\ntrade-off rather than an across-the-board improvement.'

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [51]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
errors = X_test.copy()

errors["actual"] = y_test
errors["predicted"] = test_pred
errors["probability"] = test_prob

errors["error_type"] = np.select(
    [
        (errors["actual"] == 1) & (errors["predicted"] == 1),
        (errors["actual"] == 0) & (errors["predicted"] == 0),
        (errors["actual"] == 0) & (errors["predicted"] == 1),
        (errors["actual"] == 1) & (errors["predicted"] == 0)
    ],
    [
        "true_positive",
        "true_negative",
        "false_positive",
        "false_negative"
    ],
    default="unknown"
)

print(errors["error_type"].value_counts())

error_type
true_positive     2707
false_positive    2007
true_negative      741
false_negative     545
Name: count, dtype: int64


In [52]:
false_positives = errors[
    errors["error_type"] == "false_positive"
].sort_values("probability", ascending=False)

print(
    false_positives[
        [
            "probability",
            "search_volume",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ].head(10).to_string(index=False)
)

 probability  search_volume  impressions_90d  ctr  avg_position  days_since_last_update
    0.781637           10.0               10 0.00          13.3                     305
    0.767044            NaN               10 0.00           5.0                     334
    0.750859           20.0               13 0.00          46.1                     305
    0.740469            0.0             5122 0.16          12.5                     104
    0.734861            0.0              616 0.00          22.6                     104
    0.733802            0.0             8758 0.31          13.6                     104
    0.731177           10.0               15 0.00          45.1                     305
    0.724853           10.0            12386 1.53           8.9                     104
    0.723428           70.0                2 0.00           7.0                     144
    0.723244            0.0               24 0.00           9.1                     183


In [53]:
false_negatives = errors[
    errors["error_type"] == "false_negative"
].sort_values("probability", ascending=True)

print(
    false_negatives[
        [
            "probability",
            "search_volume",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ].head(10).to_string(index=False)
)

 probability  search_volume  impressions_90d   ctr  avg_position  days_since_last_update
    0.075781            NaN                2 50.00           3.5                      20
    0.078114         1900.0           517715  0.14           4.2                     104
    0.082003           40.0           463103  0.41           2.3                      20
    0.098928           10.0              429  0.23          36.0                      20
    0.116267            0.0              200  0.00          13.8                       8
    0.164745            0.0                3 33.33           9.0                       8
    0.206266            NaN                4 25.00           2.5                      20
    0.238596           10.0                3 33.33           2.3                     151
    0.259034            NaN                5 20.00           3.4                      20
    0.278362        49500.0              988  0.00          47.3                      41


In [54]:
## 4. Errors and interpretation

"""The Logistic Regression model produced 2,707 true positives, 741 true
negatives, 2,007 false positives, and 545 false negatives on the held-out
test set.

The main error type is false positives. This is consistent with the model's
high recall of 0.8324: the model catches many of the declining pages, but it
also flags many pages that are not declining.

The false-positive examples show that the model can assign a high decline
probability to pages with very different search profiles. Some have very
little search volume and few impressions, while others have substantial
impressions. This suggests that the model is combining several weak signals
rather than relying on one simple rule.

The false-negative examples show the opposite limitation. Some pages that
actually belong to the declining class have low predicted probabilities,
including pages with high impressions and strong average positions. This
suggests that decline is not fully explained by the selected numerical
features.

Overall, the model is better suited to broad screening than to making a
final refresh decision. Human review or a later model could help reduce the
large number of false positives."""

"The Logistic Regression model produced 2,707 true positives, 741 true\nnegatives, 2,007 false positives, and 545 false negatives on the held-out\ntest set.\n\nThe main error type is false positives. This is consistent with the model's\nhigh recall of 0.8324: the model catches many of the declining pages, but it\nalso flags many pages that are not declining.\n\nThe false-positive examples show that the model can assign a high decline\nprobability to pages with very different search profiles. Some have very\nlittle search volume and few impressions, while others have substantial\nimpressions. This suggests that the model is combining several weak signals\nrather than relying on one simple rule.\n\nThe false-negative examples show the opposite limitation. Some pages that\nactually belong to the declining class have low predicted probabilities,\nincluding pages with high impressions and strong average positions. This\nsuggests that decline is not fully explained by the selected numerical\

In [55]:
coefficients = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": model.named_steps["logistic"].coef_[0]
})

coefficients["abs_coefficient"] = coefficients["coefficient"].abs()

coefficients = coefficients.sort_values(
    "abs_coefficient",
    ascending=False
)

print(coefficients.to_string(index=False))

               feature  coefficient  abs_coefficient
            word_count     0.353975         0.353975
            char_count    -0.206265         0.206265
                   ctr    -0.159906         0.159906
days_since_last_update     0.154734         0.154734
          avg_position    -0.109687         0.109687
       impressions_90d    -0.093596         0.093596
           competition     0.060239         0.060239
           scroll_rate     0.030744         0.030744
         search_volume    -0.026185         0.026185
                   cpc    -0.010190         0.010190
        ai_traffic_pct     0.008052         0.008052
       engagement_rate    -0.000367         0.000367


In [56]:
### Feature interpretation

"""The Logistic Regression coefficients show that `word_count` has the largest
positive coefficient (0.354), followed by `days_since_last_update` (0.155).
This means that, holding the other standardized features constant, higher
values of these features are associated with higher predicted probability of
decline.

`ctr` (-0.160), `char_count` (-0.206), `avg_position` (-0.110), and
`impressions_90d` (-0.094) have negative coefficients. In particular, the
negative coefficient for impressions is notable because the Week-4 baseline
used search visibility as a prioritization signal. The model therefore learns
a different relationship when other features are considered simultaneously.

The coefficients should be interpreted as associations rather than causal
effects. The relatively large coefficient for word count may also reflect
relationships between content length and other characteristics of the pages."""

'The Logistic Regression coefficients show that `word_count` has the largest\npositive coefficient (0.354), followed by `days_since_last_update` (0.155).\nThis means that, holding the other standardized features constant, higher\nvalues of these features are associated with higher predicted probability of\ndecline.\n\n`ctr` (-0.160), `char_count` (-0.206), `avg_position` (-0.110), and\n`impressions_90d` (-0.094) have negative coefficients. In particular, the\nnegative coefficient for impressions is notable because the Week-4 baseline\nused search visibility as a prioritization signal. The model therefore learns\na different relationship when other features are considered simultaneously.\n\nThe coefficients should be interpreted as associations rather than causal\neffects. The relatively large coefficient for word count may also reflect\nrelationships between content length and other characteristics of the pages.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [57]:
"""## 5. Self-check

- The model uses an 80/20 stratified train-test split with a fixed random seed.
- The held-out test set was not used to fit the model or its preprocessing.
- Missing numerical values are handled inside a training pipeline using median
  imputation.
- Features are standardized before Logistic Regression.
- `trend_direction`, `trend_pct`, and `is_declining_label` are not model
  features.
- The Logistic Regression model is compared with the Week-4 baseline on the
  same held-out test rows.
- Logistic Regression improves ROC-AUC, Average Precision, recall, and F1,
  while the Week-4 baseline has higher precision.
- Error analysis shows that false positives are the main weakness of the model.
- Feature coefficients provide an interpretable view of the model rather than
  relying only on predictive performance.
- Model complexity was not treated as evidence of improvement; performance
  was evaluated against the existing baseline."""

'## 5. Self-check\n\n- The model uses an 80/20 stratified train-test split with a fixed random seed.\n- The held-out test set was not used to fit the model or its preprocessing.\n- Missing numerical values are handled inside a training pipeline using median\n  imputation.\n- Features are standardized before Logistic Regression.\n- `trend_direction`, `trend_pct`, and `is_declining_label` are not model\n  features.\n- The Logistic Regression model is compared with the Week-4 baseline on the\n  same held-out test rows.\n- Logistic Regression improves ROC-AUC, Average Precision, recall, and F1,\n  while the Week-4 baseline has higher precision.\n- Error analysis shows that false positives are the main weakness of the model.\n- Feature coefficients provide an interpretable view of the model rather than\n  relying only on predictive performance.\n- Model complexity was not treated as evidence of improvement; performance\n  was evaluated against the existing baseline.'

In [58]:
comparison = pd.DataFrame({
    "metric": [
        "ROC-AUC",
        "Average Precision",
        "Precision",
        "Recall",
        "F1"
    ],
    "week4_baseline": [
        baseline_auc,
        baseline_ap,
        baseline_precision,
        baseline_recall,
        baseline_f1
    ],
    "logistic_regression": [
        roc_auc_score(y_test, test_prob),
        average_precision_score(y_test, test_prob),
        precision_score(y_test, test_pred),
        recall_score(y_test, test_pred),
        f1_score(y_test, test_pred)
    ]
})

print(comparison.round(4).to_string(index=False))

           metric  week4_baseline  logistic_regression
          ROC-AUC          0.5347               0.5719
Average Precision          0.5573               0.5900
        Precision          0.6214               0.5742
           Recall          0.2574               0.8324
               F1          0.3640               0.6796
